# NB2 · Train the backbones

**Safe to stop at any moment.** Runs checkpoint every epoch and resume at the
epoch they reached. Completed runs are skipped. Nothing is ever deleted. Close
the notebook whenever you like — but run the last cell first, because it is the
only thing that confirms the work is on disk and readable.

---

## What you will see while it runs

A live bar per epoch, with the numbers that matter updating **beside** it about
once a second:

```
ep 7/100  63%|███████████▌      | 738/1178 [04:12<02:31, loss=3.412, acc=0.221, img/s=402, lr=8.7e-02, vram=2.9G]
```

An epoch here is 3–35 minutes. A bar showing only position tells you the run is
alive but not whether it is *learning*, and during a multi-day programme those
are the two separate questions you actually have.

Then one line per epoch, carrying what you would otherwise have to open
`epochs.csv` to see:

```
  ep   7/100  train 22.14%  val 19.83%  top5 45.12%  loss 3.412  lr 8.66e-02  402 img/s  289s  ETA 9.3h  0.041kWh  *BEST*
```

### Warnings that appear inline, and what each one means

These are the columns that are **silent by default and unrecoverable
afterwards**, so they are surfaced while they are happening rather than left in
a CSV nobody reads by eye:

| tag | meaning | what to do |
|---|---|---|
| `[N NaN/Inf BATCHES]` | under AMP a non-finite loss is **discarded silently**. The run continues and learns nothing from those batches | a handful is normal early; hundreds means the LR is too high |
| `[N AMP OVERFLOWS]` | gradient overflows whose steps were **thrown away** | >5% of steps is a problem |
| `[LR HIGH?]` | ‖Δw‖/‖w‖ above 1e-2 | healthy is ~1e-3. Stop and check |
| `[NOT MOVING?]` | ‖Δw‖/‖w‖ below 1e-5 | nothing is learning |
| `[DATA-BOUND N%]` | the loader is the bottleneck, not the model | raise `num_workers` |

`[DATA-BOUND]` is trustworthy now: device-side augmentation is measured
separately and subtracted, so this counts genuine CPU starvation only.

---

## Run Phase 0 first. Then stop and read the gate.

**Phase 0:** `resnet50` and `vit_small_p16`, 2 seeds each. **4 runs, ~1.5 days.**

That gives one noise ceiling per family, which is the entire question:

| ρ_seed outcome | meaning | action |
|---|---|---|
| ViT below CNN by **> 0.05** | the CIFAR finding reproduces | build the atlas |
| within **±0.05** | **it was a small-data artifact** | retract the CIFAR headline; the paper becomes about scale-dependence |
| ViT **above** CNN by > 0.05 | inversion | stop, audit the measurement |
| either below **0.40** | noise-dominated at this scale | coarsen the grid, re-gate |

All four are publishable. **Row 1 flatters the existing paper, so scrutinise it
harder than the others**: check both architectures cleared the acceptance
thresholds, seed spread is under 2 points, `nan_or_inf_batches` is 0, and both
ceilings used a comparable sample count after the τ mask.

In [5]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    deefdc9590b4   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
    'ZXN0IGZvciB0aGUgb2ZmbGluZSBjaGVja3MiKQo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

msc_lib 1.0.0   torch 2.5.1+cu121
CUDA available: True
  GPU 0: NVIDIA RTX 4000 Ada Generation  20.0 GiB  sm_89


In [6]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

sess = M.Session(account='local', phase='p0', dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

storage
    C:\      715.0 GB free of   951.6
    c:\Users\Administrator\Desktop\KD\minimum-sufficient-compute\notebooks_in100   715.0 GB free of   951.6
    data    -> C:\msc_data\in100   (715 GB free, need ~26)
    results -> C:\msc_results   (715 GB free, need ~120)
    note: found an existing pack at C:\msc_data\in100
    both roots exist, are writable, and were verified by writing and reading back a probe file
[LIFE] lifecycle guard armed (SIGTERM + atexit, session limit NONE -- runs to completion)
[SESSION] account=local phase=p0 dataset=imagenet100
[SESSION] worker 0 of 1  (single worker -- set NUM_WORKERS to parallelise)
[SESSION] work=C:\msc_results  scratch=C:\msc_results
[SESSION] disk free: working=732138 MB  scratch=732138 MB
[SESSION] LOCAL-ONLY store: C:\msc_results\runs
[SESSION] nothing is uploaded and nothing is deleted. Call sess.confirm_on_disk(run_ids) before you stop.
[SESSION] offline guards active

layout under MSC_ROOT:
  runs/{run_id}/  config.yaml  summary.js

---
## Cost, from measurement rather than estimate

The plan estimated 235 GPU-hours. **Your benchmark says otherwise**, and the
shape of the answer changes what is worth running.

`vgg16` is now **45% of the entire atlas budget** for one across-CNN-family data
point. It strengthens Q3's family ordering; the Q1 headline — the reason this
replication exists — does not need it.

**You do not have to decide yet.** Phase 0 contains no `vgg16` and costs ~1.5
days. If the gap fails to reproduce, the atlas shrinks to the 2×2 anyway and
the question is moot.

Two caveats on the numbers below, both flagged in the table:

- `resnet50` and `vgg16` were measured with `cudnn.benchmark = False` — torch's
  default, and **not** what training uses (D-43). `resnet50` at 82 img/s against
  `resnet18`'s 413 is a 5× gap for 2.3× the FLOPs; expect ~180 once re-measured.
- `vit_small_p16` and `deit_small` **failed to build** in that run (D-42, fixed)
  and have never been measured. Their figures are inferred from `swin_tiny`.

In [7]:
ALL = M.zoo_for_dataset('imagenet100')
est = M.in100_estimate(ALL, seeds=3, epochs=M.IN100_EPOCHS)

print(f"{'arch':18s} {'img/s':>7s} {'s/epoch':>8s} {'h x3':>7s} {'share':>6s}  basis")
for r in est['rows']:
    print(f"{r['arch']:18s} {r['img_s']:7.0f} {r['sec_per_epoch']:8.0f} "
          f"{r['hours_all_seeds']:7.1f} {100*est['share'][r['arch']]:5.1f}%  {r['basis']}")
print()
print(f"  atlas, all 8, {M.IN100_EPOCHS} epochs: "
      f"{est['total_gpu_hours']:.0f} GPU-h = {est['days']:.1f} days")
print(f"  the plan estimated 235 -- it was optimistic by "
      f"{(est['total_gpu_hours']-235)/235*100:.0f}%")
print()
for drop in (['vgg16'], ['vgg16', 'deit_small']):
    e = M.in100_estimate([a for a in ALL if a not in drop], 3, M.IN100_EPOCHS)
    print(f"  without {drop}: {e['total_gpu_hours']:.0f} GPU-h = {e['days']:.1f} days")
for ep in (60, 80):
    e = M.in100_estimate(ALL, 3, ep)
    print(f"  all 8 at {ep} epochs: {e['total_gpu_hours']:.0f} GPU-h = {e['days']:.1f} days")
print()
print('  Dropping ONE architecture is a more honest cut than under-training')
print('  all eight: there is no published reference for this subset, so the')
print('  "these models converged" claim rests entirely on the acceptance')
print('  thresholds and has nothing to fall back on.')

arch                 img/s  s/epoch    h x3  share  basis
vgg16                   56     2121   176.7  38.7%  measured, RE-MEASURE pending (D-43)
resnet50                82     1451   120.9  26.5%  measured, RE-MEASURE pending (D-43)
convnext_tiny          272      439    36.6   8.0%  measured
swin_tiny              327      365    30.4   6.7%  measured
deit_small             380      314    26.2   5.7%  ESTIMATE -- never measured
vit_small_p16          380      314    26.2   5.7%  ESTIMATE -- never measured
resnet18               413      289    24.1   5.3%  measured
shufflenetv2_in        640      186    15.5   3.4%  measured

  atlas, all 8, 100 epochs: 457 GPU-h = 19.0 days
  the plan estimated 235 -- it was optimistic by 94%

  without ['vgg16']: 280 GPU-h = 11.7 days
  without ['vgg16', 'deit_small']: 254 GPU-h = 10.6 days
  all 8 at 60 epochs: 274 GPU-h = 11.4 days
  all 8 at 80 epochs: 365 GPU-h = 15.2 days

  Dropping ONE architecture is a more honest cut than under-training
 

---
## What to run

`PHASE = 'p0'` for the pilot, `'p1'` for the atlas. Nothing else changes.

`ARCHS` is an ordinary list — remove `vgg16` here if you take that cut.

In [8]:
PHASE  = 'p0'                     # 'p0' = pilot (4 runs) · 'p1' = atlas
EPOCHS = M.IN100_EPOCHS           # 100

if PHASE == 'p0':
    ARCHS, SEEDS = ['resnet50', 'vit_small_p16'], (1, 2)
else:
    ARCHS, SEEDS = M.zoo_for_dataset('imagenet100'), (1, 2, 3)
    # ARCHS = [a for a in ARCHS if a != 'vgg16']    # <- the 45% cut

sess = M.Session(account='local', phase=PHASE, dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0)
cfgs = [sess.config(a, seed=s, num_epochs=EPOCHS) for a in ARCHS for s in SEEDS]
run_ids = [c['run_id'] for c in cfgs]

print(f'{len(cfgs)} run(s), {EPOCHS} epochs each')
print(f"{'run_id':46s} {'opt':>6s} {'lr':>9s} {'bs':>4s} {'mixup':>6s} {'aug':>12s}")
for c in cfgs:
    print(f"{c['run_id']:46s} {c['optimizer']:>6s} {c['learning_rate']:9.5f} "
          f"{c['batch_size']:4d} {c['mixup_alpha']:6.1f} {str(c['rrc_scale']):>12s}")
e = M.in100_estimate(ARCHS, len(SEEDS), EPOCHS)
print(f"estimated {e['total_gpu_hours']:.0f} GPU-hours = {e['days']:.1f} days")
print('(an estimate; the first cell above lists which entries are measured)')

[LIFE] lifecycle guard armed (SIGTERM + atexit, session limit NONE -- runs to completion)
[SESSION] account=local phase=p0 dataset=imagenet100
[SESSION] worker 0 of 1  (single worker -- set NUM_WORKERS to parallelise)
[SESSION] work=C:\msc_results  scratch=C:\msc_results
[SESSION] disk free: working=732138 MB  scratch=732138 MB
[SESSION] LOCAL-ONLY store: C:\msc_results\runs
[SESSION] nothing is uploaded and nothing is deleted. Call sess.confirm_on_disk(run_ids) before you stop.
[SESSION] offline guards active
[DATA] found packed ImageNet-100 at C:\msc_data\in100
4 run(s), 100 epochs each
run_id                                            opt        lr   bs  mixup          aug
p0-resnet50-imagenet100-base-s1                   sgd   0.02500   64    0.0  (0.35, 1.0)
p0-resnet50-imagenet100-base-s2                   sgd   0.02500   64    0.0  (0.35, 1.0)
p0-vit_small_p16-imagenet100-base-s1            adamw   0.00006   64    0.0  (0.35, 1.0)
p0-vit_small_p16-imagenet100-base-s2            

---
## Train

Per run: claim → **dry run** → resume-or-start → train → evaluate → write
artifacts. Everything lands under `MSC_ROOT/runs/{run_id}/`.

The dry run pushes one synthetic batch through the entire path — forward, loss,
backward, optimiser step, `evaluate()`, the history row, **and a checkpoint save
and reload** — before the dataset is touched. It takes under a second and runs
*before* the run is claimed, so a broken config costs nothing and leaves no
trace in the ledger.

**Resuming is automatic.** Re-run this cell after any interruption: finished
runs are skipped, partial runs continue from their last completed epoch with
optimiser, scheduler, AMP scaler and all four RNG streams restored.

In [9]:
# `sess.train` -- NOT `M.train_backbone`. The bound method supplies hub,
# registry, work_root and data_root_out; the raw function takes them as
# required positional arguments and run_all passes only the config (D-54).
results = sess.run_all(cfgs, title='Phase 0 / atlas training')

print()
for r in results:
    if r.get('status') == 'skipped':
        print(f"  SKIPPED   {r['run_id']}  ({r.get('reason')})")
    else:
        print(f"  {r.get('status','?'):9s} {r['run_id']}  "
              f"top1={r.get('best_accuracy', float('nan')):.2f}  "
              f"{r.get('num_epochs_run','?')} epochs")

[PLAN] 1 architectures have measured timings (used for time estimates only -- ownership is fixed)

  Phase 0 / atlas training   worker 0 of 1   (stage: train, split: cost)
  universe (all runs in this phase) : 4
  my slice                          : 4   (~7.5 GPU-h estimated)
  already finished (GLOBAL, from HF): 1   <- for the 'train' stage
  MY REMAINING WORK                 : 3
--------------------------------------------------------------------------
    [mine  ] p0-resnet50-imagenet100-base-s2
    [mine  ] p0-vit_small_p16-imagenet100-base-s1
    [mine  ] p0-vit_small_p16-imagenet100-base-s2


>>> [1/3] p0-resnet50-imagenet100-base-s2
[DRY] backbone dry run ok (1.11s, 224px, 100 classes)
[CLAIM] p0-resnet50-imagenet100-base-s2 was left 'paused' by an earlier session of local 5 min ago -- resuming it. If you genuinely have two live sessions on this account, give them different WORKER_IDs.
[CLAIM] claiming p0-resnet50-imagenet100-base-s2 (resuming own run from a previous session (5 

ep 69/100:   0%|          | 0/1866 [00:00<?, ?b/s]

  ep  69/100  train 96.87%  val 80.35%  top5 94.45%  loss 0.908  lr 6.01e-03  80 img/s  1502s  ETA 12.9h  0.045kWh
[HF] pushed at epoch 69 (elapsed 0.4 h)


ep 70/100:   0%|          | 0/1866 [00:00<?, ?b/s]

  ep  70/100  train 96.64%  val 80.42%  top5 94.50%  loss 0.913  lr 5.66e-03  80 img/s  1500s  ETA 12.4h  0.045kWh  [LR HIGH?]
[HF] pushed at epoch 70 (elapsed 0.8 h)


...

ep 73/100:   0%|          | 0/1866 [00:00<?, ?b/s]

  ep  73/100  train 97.80%  val 59.41%  top5 79.62%  loss 0.842  lr 1.17e-05  605 img/s  203s  ETA 1.5h  0.007kWh  [LR HIGH?]


ep 74/100:   0%|          | 0/1866 [00:00<?, ?b/s]

[STOP] p0-vit_small_p16-imagenet100-base-s2 interrupted -- immediate push
[HF] disabled
[STOP] interrupted -- everything flushed to HF; re-run to resume


KeyboardInterrupt: 

---
## Before you stop — confirm the work is on disk

`confirm_on_disk` **opens every required artifact**. Stronger than a presence
check: a run whose `summary.json` exists but whose `epochs.csv` is zero bytes
looks healthy to a presence check and fails during analysis weeks later.

- **complete** — every required artifact present, non-empty, parseable
- **resumable** — `ckpt_last.pt` is there. **Safe to stop.** Being unfinished is
  the normal state of a paused run, not a failure
- **at risk** — missing, zero-byte, or corrupt

In [ ]:
status = sess.confirm_on_disk(run_ids)

print()
if status['at_risk']:
    print('  *** Do not treat the AT RISK runs as done. Re-run the training')
    print('  *** cell; finished work is skipped and unfinished work resumes.')
else:
    print('  Nothing is at risk.')
    print('  Next: NB3_Measure, then NB4_Analysis.')
    if PHASE == 'p0':
        print()
        print('  THEN COME BACK AND READ THE GATE at the top of this notebook')
        print('  before starting the atlas. Phase 0 is 8% of the programme and')
        print('  it decides whether the other 92% is worth spending.')